In [1]:
from datasets import load_dataset

dataset_dict = load_dataset("SetFit/ag_news", split='train', download_mode="reuse_dataset_if_exists")
remove_columns = dataset_dict.column_names
raw_labels = dataset_dict.unique('label')
label2id = {str(label): idx for idx, label in enumerate(sorted(map(str, raw_labels)))}
id2label = {idx: label for label, idx in label2id.items()}

/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
from copy import deepcopy

from shadow_peft import ShadowForSequenceClassification
from transformers import AutoModelForSequenceClassification, AutoTokenizer

shadow_peft_path = "shadow-llm/agnews_shadow_peft"
tokenizer = AutoTokenizer.from_pretrained(shadow_peft_path)

base_model = AutoModelForSequenceClassification.from_pretrained(
        "Qwen/Qwen3-0.6B",
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    ).to('cuda:0')

model = ShadowForSequenceClassification.from_pretrained(
    deepcopy(base_model), shadow_peft_path, is_trainable=False, inference_mode='base_shadow').to(base_model.device)
shadow_only = ShadowForSequenceClassification.from_pretrained(
    deepcopy(base_model), shadow_peft_path, is_trainable=False, inference_mode='shadow_only').to(base_model.device)

model = model.to(base_model.dtype)
model = model.eval()
shadow_only = shadow_only.to(base_model.dtype)
shadow_only = shadow_only.eval()


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7072.53it/s]
Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 49734.83it/s]


In [3]:
import time

import torch


doc = '''
Owners Seek Best Ballpark Deal for Expos (AP) AP - Trying to get the best possible ballpark deal for the Montreal Expos, major league baseball instructed its lawyers to press ahead with negotiations involving four of the areas bidding for the team.
'''

tok = tokenizer(doc, return_tensors='pt')
tok = tok.to(model.device)
s_time = time.time()
with torch.no_grad():
    outputs = model(**tok)
print('Inference time:', time.time() - s_time)
prediction = outputs.logits.argmax().item()
shadow_prediction = outputs.shadow_logits.argmax().item()

id2label[prediction], id2label[shadow_prediction]

Inference time: 0.33975982666015625


('1', '1')

In [4]:
s_time = time.time()
with torch.no_grad():
    outputs = shadow_only(**tok)
print('Inference time:', time.time() - s_time)
prediction = outputs.logits.argmax().item()
shadow_prediction = outputs.shadow_logits.argmax().item()

id2label[shadow_prediction]

Inference time: 0.004050254821777344


'1'